Import packages

In [321]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import joblib

## Recommendation System and NearestNeighbors, Cosine_similarity

### Load the Dataset

### Orginal Dataframe without Encoding:

In [322]:
global_mobile_review_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\data\processed_data\Mobile_Reviews_Sentiment_before_encoding.csv")


#### Feature selection:


In [418]:
product_df = (
            global_mobile_review_df.groupby(["brand", "model"], as_index=False)
            .agg({
                "price_usd": "mean",
                "rating": "mean",
                "battery_life_rating": "mean",
                "camera_rating": "mean",
                "performance_rating": "mean",
                "design_rating": "mean",
                "display_rating": "mean"
            })
        )

product_df = product_df.round(2)
print(product_df.shape)

joblib.dump(product_df, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\product_df.pkl")
print(" product_df is saved as pkl file")

(22, 9)
 product_df is saved as pkl file


#### One Hot Encoding the brand and model:

In [419]:
product_encoded = pd.get_dummies(
                                    product_df,
                                    columns=["brand"],
                                    dtype=int
)

product_encoded.head()

,model,price_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,brand_Apple,brand_Google,brand_Motorola,brand_OnePlus,brand_Realme,brand_Samsung,brand_Xiaomi
0,iPhone 13,1108.26,3.13,2.75,2.75,2.77,2.77,2.71,1,0,0,0,0,0,0
1,iPhone 14,1099.26,3.15,2.74,2.78,2.72,2.77,2.76,1,0,0,0,0,0,0
2,iPhone 15 Pro,1103.40,3.06,2.68,2.71,2.66,2.68,2.69,1,0,0,0,0,0,0
3,iPhone SE,1103.39,3.08,2.73,2.70,2.73,2.69,2.74,1,0,0,0,0,0,0
4,Pixel 6,807.92,3.09,2.71,2.71,2.69,2.71,2.72,0,1,0,0,0,0,0


In [420]:
mobile_names = product_df[["brand","model"]]

In [433]:
X = product_encoded.drop(columns=["model"])

# X = product_encoded

print(product_encoded.shape)


(22, 15)


In [422]:
print(X.columns)

Index(['price_usd', 'rating', 'battery_life_rating', 'camera_rating',
       'performance_rating', 'design_rating', 'display_rating', 'brand_Apple',
       'brand_Google', 'brand_Motorola', 'brand_OnePlus', 'brand_Realme',
       'brand_Samsung', 'brand_Xiaomi'],
      dtype='str')


Scaling the features:

In [432]:
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# scaler = Normalizer()
# X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

joblib.dump(X_scaled, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\X_scaled.pkl")
print("Scalar is saved as pkl file")


(22, 14)
Scalar is saved as pkl file


#### Train the Model:

##### Using NearestNeighbors:

In [424]:
recommend_model = NearestNeighbors( n_neighbors=6,
                                    metric="cosine",
                                    algorithm="brute"
)

recommend_model.fit(X_scaled)
print(recommend_model)

joblib.dump(recommend_model, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\recommend_model.pkl")
print(" recommend model is saved as pkl file")

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)
 recommend model is saved as pkl file


##### Using cosine_similarity:

In [425]:
# similarity = cosine_similarity(X_scaled)

# print(similarity.shape)

# joblib.dump(similarity, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\models\similarity.pkl")
# print(" similarity is saved as pkl file")

In [426]:
# print(similarity.shape)

# print(similarity[0][:10])

#### Create Recommendation Function:

#### Based on NearestNeighbors:

In [ ]:
def recommend_mobile(model_name):

    index = product_df[product_df["model"] == model_name].index[0]

    distances, indices = recommend_model.kneighbors(
                                X_scaled[index].reshape(1, -1) )

    # Get full recommendations first
    recommendations = product_df.iloc[indices[0][1:]]
    [
                                    ["brand", "model", "price_usd", "rating"]].copy()

    # Add distance
    recommendations["distance"] = distances[0][1:]

    # Add similarity percentage:    
    recommendations["similarity"] = (1 - recommendations["distance"]) * 100

    return recommendations.sort_values("similarity", ascending=False)

#### Based on cosine_similarity:

In [428]:
# def recommend_mobile(model_name, top_n=5):

#     # Find selected mobile index
#     index = product_df[ product_df["model"] == model_name ].index[0]

#     # Get similarity scores
#     similarity_scores = list(enumerate(similarity[index]))

#     # Sort by similarity (highest first)
#     similarity_scores = sorted( similarity_scores, key=lambda x: x[1], reverse=True )

#     # Remove the selected mobile itself
#     similarity_scores = similarity_scores[1:top_n+1]

#     # Get indices
#     mobile_indices = [i[0] for i in similarity_scores]

#     # Fetch recommendations
#     recommendations = product_df.iloc[mobile_indices].copy()

#     # Add similarity percentage
#     recommendations["Similarity (%)"] = [ round(score * 100, 2) for _, score in similarity_scores
#     ]

#     return recommendations[
#         [
#             "brand","model","price_usd","rating",
#             "Similarity (%)"
#         ]
#     ]

#### Test

In [429]:
recommend_mobile("Pixel 6")

,brand,model,price_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,distance,similarity
6,Google,Pixel 8,798.97,3.10,2.73,2.72,2.69,2.74,2.74,0.018229,98.177140
5,Google,Pixel 7a,806.82,3.11,2.72,2.73,2.73,2.72,2.75,0.021562,97.843832
1,Apple,iPhone 14,1099.26,3.15,2.74,2.78,2.72,2.77,2.76,0.286819,71.318139
0,Apple,iPhone 13,1108.26,3.13,2.75,2.75,2.77,2.77,2.71,0.320075,67.992475
16,Samsung,Galaxy Note 20,899.73,3.13,2.74,2.73,2.76,2.75,2.73,0.323305,67.669483


In [431]:
print(global_mobile_review_df["rating"].describe())


count    50000.000000
mean         3.102160
std          1.248661
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64
